In [1]:
%load_ext autoreload
%autoreload 2

# Predict with the Flow Matching Model or CGM

In [ ]:
import importlib

import hydra
import lightning as L
import numpy as np
import torch
import xarray as xr
from einops import rearrange, reduce
from omegaconf import DictConfig
from tqdm import tqdm

from genpp import BASE_DIR
from genpp.configs import register_resolvers
from genpp.data import FORECAST_ENS_FLAT_AGG_PATH, OBSERVATIONS_FLAT_PATH
from genpp.eval.utils import log_scores, update_wandb_run
from genpp.models.cgm.chen import CNNChenModel
from genpp.models.loss import EnergyScore, EnsembleCRPS, VariogramScore

try:
    register_resolvers()
except Exception:
    pass

torch.set_float32_matmul_precision("high")

## Best Models
FM model: `feik/genpp/blkpcik8`

Chen model: `feik/genpp/qbuvhf5p`


In [3]:
# Model ID is generated by WandB
RUN_PATH = "feik/genpp/qbuvhf5p"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

SCORE_FILE = MODEL_DIR / "scores.csv"

In [4]:
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
    # Do not shuffle any dataloader
    cfg.data.module.dataloader_config.train.shuffle = False
    cfg.data.module.dataloader_config.val.shuffle = False
    cfg.data.module.dataloader_config.val.batch_size = 16  # For faster predictions
    cfg.data.module.dataloader_config.test.shuffle = False
    datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup(stage="validate")

Configuration hash: 05a2d862df574c3ec09a73a2516597b76945436332ad35785c3a5fe80dbc1c8c
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_05a2d862df574c3ec09a73a2516597b76945436332ad35785c3a5fe80dbc1c8c.pt.


In [25]:
class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

if ModelClass is CNNChenModel:
    model = ModelClass.load_from_checkpoint(
        MODEL_CHECKPOINT,
        final_activation=hydra.utils.instantiate(cfg.model.final_activation),
        loss_fn=hydra.utils.instantiate(cfg.model.loss_fn),
    )
else:
    model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)

Loading scale_variance_td from checkpoint


/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'final_activation' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['final_activation'])`.
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'loss_fn' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss_fn'])`.


In [26]:
limit_predict_batches = None
trainer = L.Trainer(
    logger=False, accelerator="gpu", devices=[0], limit_predict_batches=limit_predict_batches
)
pred_list = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)
predictions = torch.cat(pred_list, dim=0)  # type: ignore

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 227/227 [00:13<00:00, 17.10it/s]


In [28]:
# Rescale the predictions (TODO this should happen in the predict step)
reverse_transform = datamodule.y_reverseModules[0]
mean = rearrange(reverse_transform.mean, "f -> 1 1 f 1 1")
scale = rearrange(reverse_transform.scale, "f -> 1 1 f 1 1")

predictions_rescaled = predictions * scale + mean

In [30]:
import pickle

x_ds = xr.open_dataset(FORECAST_ENS_FLAT_AGG_PATH)

time_slice = {
    "train": hydra.utils.instantiate(cfg.data.splits.train),
    "val": hydra.utils.instantiate(cfg.data.splits.val),
    "test": hydra.utils.instantiate(cfg.data.splits.test),
}
val_slice = time_slice["val"]

x_ds = x_ds.stack(sample=("time", "prediction_timedelta"))

# Lets investigate the len of the dataset splits
with open(datamodule.metadata_path, "rb") as f:
    cache_metadata = pickle.load(f)

train_idx = cache_metadata["split_indices"]["train"]
val_idx = cache_metadata["split_indices"]["val"]
test_idx = cache_metadata["split_indices"]["test"]

# Now lets get the actual times we need for the val set and and get the val data
val_samples = x_ds.isel(sample=val_idx).sample
val_times = val_samples.time + val_samples.prediction_timedelta
val_times = val_times.compute().values

# Now load the y_val data for the val times
y_val = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(time=val_times)
    .to_dataarray("feature")
    .transpose("time", "feature", "longitude", "latitude")
)
feature_order = list(cfg.data.y_select_variables)
y_val = y_val.sel(feature=feature_order)
y_t = torch.from_numpy(y_val.values).to(predictions_rescaled)

In [31]:
SKIP_VARIOGRAM = True  # Option to skip variogram score computation due to long computation times

# Compute the scores
crps_ens = EnsembleCRPS()
es = EnergyScore(clamp=False)
vs = VariogramScore(p=0.5)

crps_per_margin = crps_ens(predictions_rescaled, y_t)

# Rearrange so that we compute the scores seperatly for each variable
x_spatial = rearrange(predictions_rescaled, "t n d lat lon -> t d n (lat lon)")
y_spatial = rearrange(y_t, "t d lat lon -> t d (lat lon)")

# _u is for un-reduced scores
energy_score_per_var_u = es(x_spatial, y_spatial)

# For the VS there are huge intermediary results, thats why it is computed batchwise
if not SKIP_VARIOGRAM:
    vss = []
    for x_i, y_i in tqdm(zip(x_spatial, y_spatial), total=predictions_rescaled.shape[0]):
        vss.append(vs(x_i, y_i))
    variogram_score_per_var_u = torch.stack(vss)


# Rearrange to compute scores for both variables together
x_full = rearrange(predictions_rescaled, "t n d lat lon -> t n (d lat lon)")
y_full = rearrange(y_t, "t d lat lon -> t (d lat lon)")

energy_score_full_u = es(x_full, y_full)
if not SKIP_VARIOGRAM:
    vss = []
    for x_i, y_i in tqdm(zip(x_full, y_full), total=predictions_rescaled.shape[0]):
        vss.append(vs(x_i, y_i))
    variogram_score_full_u = torch.stack(vss)

In [32]:
# Reduce the scores
crps_per_var = reduce(crps_per_margin, "t d h w -> d", reduction="mean")
crps_full = reduce(crps_per_margin, "t d h w -> 1", "mean")
energy_score_per_var = reduce(energy_score_per_var_u, "t d -> d", "mean")
energy_score_full = reduce(energy_score_full_u, "t -> 1", "mean")
if not SKIP_VARIOGRAM:
    variogram_score_per_var = reduce(variogram_score_per_var_u, "t d -> d", "mean")
    variogram_score_full = reduce(variogram_score_full_u, "t -> 1", "mean")

In [34]:
# Log the Scores
model_class = cfg.model._target_.split(".")[-1]


log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="CRPS",
    variables=datamodule.y_select_variables,
    scores=crps_per_var,
)
log_scores(
    file=SCORE_FILE, model=model_class, metric="CRPS", variables=["combined"], scores=crps_full
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=energy_score_per_var,
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="EnergyScore",
    variables=["combined"],
    scores=energy_score_full,
)
if not SKIP_VARIOGRAM:
    log_scores(
        file=SCORE_FILE,
        model=model_class,
        metric="VariogramScore",
        variables=datamodule.y_select_variables,
        scores=variogram_score_per_var,
    )
    # Log full scores
    log_scores(
        file=SCORE_FILE,
        model=model_class,
        metric="VariogramScore",
        variables=["combined"],
        scores=variogram_score_full,
    )

In [47]:
# Now also compute the scores per leadtime
val_timedeltas = val_samples.prediction_timedelta.values
unique_timedeltas = np.unique(val_timedeltas)

scores = {}
for td in sorted(unique_timedeltas):
    mask = val_timedeltas == td
    td_hours = td / np.timedelta64(1, "h")
    print(f"Processing leadtime {td_hours:.0f}h with {np.sum(mask)} samples")
    masked_crps_per_margin = reduce(crps_per_margin[mask], "t d h w -> d", reduction="mean")
    masked_crps_full = reduce(crps_per_margin[mask], "t d h w -> 1", reduction="mean")
    masked_energy_score_per_var = reduce(energy_score_per_var_u[mask], "t d -> d", "mean")
    masked_energy_score_full = reduce(energy_score_full_u[mask], "t -> 1", "mean")
    if not SKIP_VARIOGRAM:
        masked_variogram_score_per_var = reduce(variogram_score_per_var_u[mask], "t d -> d", "mean")
        masked_variogram_score_full = reduce(variogram_score_full_u[mask], "t -> 1", "mean")

    # Create scores dict for this leadtime
    leadtime_scores = {
        "CRPS_combined": masked_crps_full.item(),
        "CRPS_2m_temperature": masked_crps_per_margin[0].item(),
        "CRPS_10m_windspeed": masked_crps_per_margin[1].item(),
        "EnergyScore_combined": masked_energy_score_full.item(),
        "EnergyScore_2m_temperature": masked_energy_score_per_var[0].item(),
        "EnergyScore_10m_windspeed": masked_energy_score_per_var[1].item(),
    }
    if not SKIP_VARIOGRAM:
        leadtime_scores.update(
            {
                "VariogramScore_combined": masked_variogram_score_full.item(),  # type: ignore
                "VariogramScore_2m_temperature": masked_variogram_score_per_var[0].item(),  # type: ignore
                "VariogramScore_10m_windspeed": masked_variogram_score_per_var[1].item(),  # type: ignore
            }
        )
    scores[f"{td_hours:.0f}h"] = leadtime_scores

# Transpose the scores dict for logging
transposed = {}
for outer_key, inner_dict in scores.items():
    for inner_key, value in inner_dict.items():
        if inner_key not in transposed:
            transposed[inner_key] = {}
        transposed[inner_key][outer_key] = value
scores = transposed
scores

Processing leadtime 24h with 724 samples
Processing leadtime 48h with 724 samples
Processing leadtime 72h with 724 samples
Processing leadtime 96h with 724 samples
Processing leadtime 120h with 724 samples


{'CRPS_combined': {'24h': 2.3668086528778076,
  '48h': 2.312922716140747,
  '72h': 2.2646427154541016,
  '96h': 2.1740920543670654,
  '120h': 2.060359477996826},
 'CRPS_2m_temperature': {'24h': 3.1540586948394775,
  '48h': 3.096851348876953,
  '72h': 3.0601940155029297,
  '96h': 2.9703993797302246,
  '120h': 2.805785655975342},
 'CRPS_10m_windspeed': {'24h': 1.5795578956604004,
  '48h': 1.528991937637329,
  '72h': 1.469092607498169,
  '96h': 1.377784252166748,
  '120h': 1.3149300813674927},
 'EnergyScore_combined': {'24h': 143.27392578125,
  '48h': 139.8710174560547,
  '72h': 136.80740356445312,
  '96h': 131.36996459960938,
  '120h': 124.22945404052734},
 'EnergyScore_2m_temperature': {'24h': 121.45957946777344,
  '48h': 119.0104751586914,
  '72h': 117.28422546386719,
  '96h': 113.72278594970703,
  '120h': 107.51122283935547},
 'EnergyScore_10m_windspeed': {'24h': 66.46013641357422,
  '48h': 64.21897888183594,
  '72h': 61.483646392822266,
  '96h': 57.600337982177734,
  '120h': 55.11286

In [49]:
# Update WandB run
update_wandb_run(f"feik/genpp/{MODEL_ID}", {"val": scores})